# Arabic LLM Fine-Tuning: Qwen2.5-7B Domain Adaptation
**Task**: Arabic Cultural & Islamic QA (PalmX 2025 Benchmark)
**Methodology**: 4-bit NormalFloat QLoRA + SFT Prompt Loss Masking + Fast Logit Likelihood Scoring
**Target Baseline**: NileChat-3B (69.5% accuracy)

In [ ]:
!pip install -q bitsandbytes transformers datasets peft torch accelerate scikit-learn pandas tqdm

In [ ]:
import os
import gc
import json
import logging
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("ArabicLLMFineTuner")

In [ ]:
class ArabicLLMFineTuner:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct", token=None):
        self.model_id = model_id
        self.token = token or os.getenv("HF_TOKEN")
        self.model = None
        self.tokenizer = None
        self.peft_model = None

    def setup_model_and_tokenizer(self, use_bf16=True):
        """Setup base model with 4-bit NF4 quantization and right-padded tokenizer"""
        logger.info(f"Loading model: {self.model_id}")
        compute_dtype = torch.bfloat16 if (use_bf16 and torch.cuda.is_bf16_supported()) else torch.float16

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            token=self.token,
            torch_dtype=compute_dtype,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            token=self.token
        )

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.tokenizer.padding_side = "right"
        self.model = prepare_model_for_kbit_training(self.model)
        logger.info("Model and tokenizer initialized.")

    def setup_improved_lora_config(self, r=32, lora_alpha=64, lora_dropout=0.05):
        """Configure LoRA without saving full embeddings to maintain compact adapter size"""
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
            bias="none",
        )
        self.peft_model = get_peft_model(self.model, lora_config)
        self.peft_model.print_trainable_parameters()
        return lora_config

    def load_and_prepare_data(self, task="subtask2", validation_split=0.2, augment=False):
        """Load dataset with caching safeguards, stratified split, and optional permutation augmentation"""
        dataset_name = "UBC-NLP/palmx_2025_subtask1_culture" if task == "subtask1" else "UBC-NLP/palmx_2025_subtask2_islamic"
        try:
            train_data = load_dataset(dataset_name, split="train", download_mode="force_redownload")
        except Exception:
            train_data = load_dataset(dataset_name, split="train")

        try:
            eval_data = load_dataset(dataset_name, split="dev")
            logger.info(f"Using existing dev split: {len(eval_data)} samples")
        except Exception:
            logger.info(f"Creating stratified {validation_split*100:.0f}% validation split...")
            train_eval_split = train_data.train_test_split(
                test_size=validation_split,
                seed=42,
                shuffle=True,
                stratify_by_column="answer"
            )
            train_data = train_eval_split['train']
            eval_data = train_eval_split['test']

        if augment:
            import datasets
            aug_records = []
            labels = ["A", "B", "C", "D"]
            for item in train_data:
                aug_records.append(dict(item))
                opts = [item["A"], item["B"], item["C"], item["D"]]
                ans_idx = labels.index(item["answer"])
                shifted_opts = opts[1:] + opts[:1]
                new_ans_idx = (ans_idx - 1) % 4
                aug_records.append({
                    "id": f"{item.get('id', 's')}_perm1",
                    "question": item["question"],
                    "A": shifted_opts[0], "B": shifted_opts[1], "C": shifted_opts[2], "D": shifted_opts[3],
                    "answer": labels[new_ans_idx]
                })
            train_data = datasets.Dataset.from_list(aug_records)
            logger.info(f"Augmented training size: {len(train_data)}")

        logger.info(f"Split - Train: {len(train_data)}, Validation: {len(eval_data)}")
        return train_data, eval_data

    def prepare_dataset_improved(self, dataset, max_length=1024, mask_prompt=True):
        """Tokenize with prompt loss masking (labels=-100 on prompt tokens)"""
        logger.info(f"Preparing dataset (prompt_loss_masking={mask_prompt})...")
        formatted_examples = []

        for i, item in enumerate(tqdm(dataset, desc="Tokenizing")):
            try:
                prompt_text = f"""<|im_start|>system
أنت خبير في الثقافة الإسلامية. أجب على السؤال متعدد الخيارات بتقديم حرف الإجابة الصحيحة فقط (A، B، C، أو D).<|im_end|>
<|im_start|>user
السؤال: {item['question']}

A. {item['A']}
B. {item['B']}
C. {item['C']}
D. {item['D']}

الجواب:<|im_end|>
<|im_start|>assistant
"""
                target_text = f"{item['answer']}<|im_end|>"

                if mask_prompt:
                    prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
                    target_ids = self.tokenizer.encode(target_text, add_special_tokens=False)
                    full_ids = prompt_ids + target_ids
                    if len(full_ids) > max_length:
                        full_ids = full_ids[:max_length]
                    num_prompt = min(len(prompt_ids), len(full_ids))
                    labels = [-100] * num_prompt + full_ids[num_prompt:]
                else:
                    tokenized = self.tokenizer(prompt_text + target_text, truncation=True, max_length=max_length)
                    full_ids = tokenized["input_ids"]
                    labels = list(full_ids)

                formatted_examples.append({
                    "input_ids": full_ids,
                    "attention_mask": [1] * len(full_ids),
                    "labels": labels
                })
            except Exception as e:
                logger.error(f"Error processing sample {i}: {e}")
                continue

        return formatted_examples

    def create_improved_data_collator(self):
        """Dynamic batch padding with label=-100 padding"""
        def data_collator(features):
            max_len = max(len(f["input_ids"]) for f in features)
            batch = {"input_ids": [], "attention_mask": [], "labels": []}
            for f in features:
                pad_len = max_len - len(f["input_ids"])
                batch["input_ids"].append(f["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
                batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
                batch["labels"].append(f["labels"] + [-100] * pad_len)
            return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}
        return data_collator

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=-1)
        mask = labels != -100
        acc = (preds[mask] == labels[mask]).mean()
        return {"accuracy": float(acc), "eval_samples": int(mask.sum())}

    def fine_tune_improved(self, train_data, eval_data, output_dir="./arabic_llm_qwen_finetuned"):
        train_dataset = self.prepare_dataset_improved(train_data, mask_prompt=True)
        eval_dataset = self.prepare_dataset_improved(eval_data, mask_prompt=True)
        has_bf16 = torch.cuda.is_bf16_supported()

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=5,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=4,
            gradient_accumulation_steps=8,
            warmup_ratio=0.1,
            learning_rate=1e-4,
            weight_decay=0.01,
            bf16=has_bf16,
            fp16=not has_bf16,
            logging_steps=25,
            save_steps=100,
            eval_steps=100,
            eval_strategy="steps",
            save_strategy="steps",
            load_best_model_at_end=True,
            metric_for_best_model="eval_accuracy",
            greater_is_better=True,
            remove_unused_columns=False,
            dataloader_pin_memory=False,
            gradient_checkpointing=True,
            report_to="none",
            save_total_limit=3,
            lr_scheduler_type="cosine",
            max_grad_norm=1.0,
        )

        trainer = Trainer(
            model=self.peft_model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=self.create_improved_data_collator(),
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        logger.info("Starting improved fine-tuning...")
        trainer.train()
        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        logger.info(f"Fine-tuning completed! Saved to {output_dir}")
        return trainer

    def evaluate_with_baseline_format(self, eval_data, max_length=1024):
        """Fast Single-pass Logit Extraction (4x faster than multiple full-text passes)"""
        self.peft_model.eval()
        correct_predictions = 0
        total_questions = len(eval_data)
        submission_data = []

        choice_tokens = [self.tokenizer.encode(f" {c}", add_special_tokens=False)[-1] for c in ['A', 'B', 'C', 'D']]
        logger.info(f"Evaluating on {total_questions} validation samples via fast single forward pass...")

        for i, example in enumerate(tqdm(eval_data, desc="Evaluating")):
            try:
                prompt = f"{example['question']}\n\nA. {example['A']}\nB. {example['B']}\nC. {example['C']}\nD. {example['D']}\n\nالجواب:"
                inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(self.peft_model.device)

                with torch.no_grad():
                    logits = self.peft_model(**inputs).logits[0, -1, :]
                    choice_scores = [logits[tok_id].item() for tok_id in choice_tokens]

                predicted_idx = int(np.argmax(choice_scores))
                predicted_answer = ['A', 'B', 'C', 'D'][predicted_idx]
                is_correct = predicted_answer == example["answer"]

                submission_data.append({
                    "id": example.get("id", f"sample_{i}"),
                    "prediction": predicted_answer,
                    "correct_answer": example["answer"],
                    "is_correct": is_correct,
                    "scores": choice_scores
                })

                if is_correct:
                    correct_predictions += 1
            except Exception as e:
                logger.error(f"Error on sample {i}: {e}")
                submission_data.append({
                    "id": example.get("id", f"sample_{i}"),
                    "prediction": "A",
                    "correct_answer": example["answer"],
                    "is_correct": False,
                    "scores": [0, 0, 0, 0]
                })

            if i % 100 == 0:
                torch.cuda.empty_cache()

        accuracy = (correct_predictions / total_questions) * 100.0
        logger.info(f"\n=== Accuracy: {accuracy:.2f}% ({correct_predictions}/{total_questions}) ===")
        return submission_data, accuracy

    def analyze_errors(self, results, eval_data):
        error_analysis = {'by_answer': {'A': 0, 'B': 0, 'C': 0, 'D': 0}, 'confusion_matrix': {}, 'difficult_questions': []}
        for i, r in enumerate(results):
            if not r['is_correct']:
                c_ans = r['correct_answer']
                p_ans = r['prediction']
                error_analysis['by_answer'][c_ans] = error_analysis['by_answer'].get(c_ans, 0) + 1
                if c_ans not in error_analysis['confusion_matrix']:
                    error_analysis['confusion_matrix'][c_ans] = {}
                error_analysis['confusion_matrix'][c_ans][p_ans] = error_analysis['confusion_matrix'][c_ans].get(p_ans, 0) + 1
                scores = sorted(r['scores'])
                if (scores[-1] - scores[-2]) < 1.0:
                    error_analysis['difficult_questions'].append({
                        'id': r['id'],
                        'question': eval_data[i]['question'][:200] + '...',
                        'confidence': float(scores[-1] - scores[-2]),
                        'predicted': p_ans,
                        'correct': c_ans
                    })
        return error_analysis

# Alias for backwards compatibility
ImprovedQwenFineTuner = ArabicLLMFineTuner

def main_improved_finetune():
    ft = ArabicLLMFineTuner(model_id="Qwen/Qwen2.5-7B-Instruct")
    try:
        ft.setup_model_and_tokenizer(use_bf16=True)
        ft.setup_improved_lora_config(r=32, lora_alpha=64)
        train_data, eval_data = ft.load_and_prepare_data(task="subtask2", validation_split=0.2, augment=False)
        trainer = ft.fine_tune_improved(train_data, eval_data)
        results, accuracy = ft.evaluate_with_baseline_format(eval_data)
        error_analysis = ft.analyze_errors(results, eval_data)
        pd.DataFrame(results).to_csv(f"improved_results_{accuracy:.1f}acc.csv", index=False)
        with open(f"error_analysis_{accuracy:.1f}acc.json", "w", encoding="utf-8") as f:
            json.dump(error_analysis, f, indent=2, ensure_ascii=False)
        logger.info(f"Baseline: 69.5% | Achieved Accuracy: {accuracy:.2f}%")
    finally:
        torch.cuda.empty_cache()
        gc.collect()

if __name__ == "__main__":
    main_improved_finetune()

In [ ]:
from huggingface_hub import login
# Log in securely using an environment variable or interactive prompt
token = os.getenv("HF_TOKEN")
if token:
    login(token=token)
else:
    from huggingface_hub import notebook_login
    notebook_login()